<h3>SciPy</h3>

### Що таке SciPy


SciPy (Scientific Python) — це бібліотека для Python, яка надає набір інструментів для виконання наукових та інженерних обчислень. SciPy побудована на основі бібліотеки NumPy та складається з кількох модулів, із яких нам буде потрібний модуль для статистичних обчислень — scipy.stats.

scipy.stats надає широкий спектр функцій для статистичного аналізу, наприклад:

* Функції для роботи за статистичними розподілами;
* Функції описової статистики;
* Функції для тестування гіпотез тощо.


За допомогою функцій тестування гіпотез із модуля scipy.stats ми маємо змогу оцінити статистичну значущість результатів нашого A/B тесту.

In [2]:
import pandas as pd
import numpy as np

### Контрольна група

* Кількість користувачів у групі: 7 015
* Кількість конверсій у групі: 139
* Значення конверсії: 1.98%


### Альтернативна група

* Кількість користувачів у групі: 6 987
* Кількість конверсій у групі: 314
* Значення конверсії: 4.49%

In [3]:
test_data = pd.DataFrame(data={'test_group': ['basic']*7015 + ['test']*6987,
                               'conversion': [1]*139 + [0]*(7015-139) + [1]*314 + [0]*(6987-314)})

print(test_data)
# Перевіримо, чи все коректно згенерували:
print(test_data.groupby('test_group').describe())

      test_group  conversion
0          basic           1
1          basic           1
2          basic           1
3          basic           1
4          basic           1
...          ...         ...
13997       test           0
13998       test           0
13999       test           0
14000       test           0
14001       test           0

[14002 rows x 2 columns]
           conversion                                             
                count      mean       std  min  25%  50%  75%  max
test_group                                                        
basic          7015.0  0.019815  0.139373  0.0  0.0  0.0  0.0  1.0
test           6987.0  0.044941  0.207189  0.0  0.0  0.0  0.0  1.0


### Критерій Хі-квадрат



Критерій Хі-квадрат, або Chi-squared — ще один із популярних критеріїв тестування гіпотез, призначений для роботи з категоріальними даними. Цей критерій перевіряє залежність змінної від категорії, тобто у випадку A/B тесту — чи залежить метрика, яку ми аналізуємо, від тестової групи, в яку потрапив користувач.



У SciPy Критерій Хі-квадрат представлений функцією stats.chi2_contingency.



### Нульова гіпотеза: змінна є незалежною від категорії, під яку вона підпадає.



Основні параметри функції:

* observed — контингентна таблиця, що містить спостережені частоти (тобто кількість випадків) у кожній категорії. По суті, це зведена таблиця, в якій рядок — це категорія, стовпчик — значення змінної, а значення в таблиці — кількість спостережень.

In [10]:
from scipy import stats

test_data = pd.DataFrame(data={'test_group': ['basic']*7015 + ['test']*6987,
                               'conversion': [1]*139 + [0]*(7015-139) + [1]*314 + [0]*(6987-314)})

# Перевіримо, чи все коректно згенерували:
print(test_data.groupby('test_group').describe())

alpha = 0.05

observed = pd.crosstab(test_data['test_group'].values, test_data['conversion'].values)

print(observed)

print(stats.chi2_contingency(observed))

statistic, p_value, dof, expected_values = stats.chi2_contingency(observed)

print(f'chi2-statistic: {round(statistic, 2)}, p-value: {round(pvalue, 2)}')

if p_value < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')

           conversion                                             
                count      mean       std  min  25%  50%  75%  max
test_group                                                        
basic          7015.0  0.019815  0.139373  0.0  0.0  0.0  0.0  1.0
test           6987.0  0.044941  0.207189  0.0  0.0  0.0  0.0  1.0
col_0     0    1
row_0           
basic  6876  139
test   6673  314
Chi2ContingencyResult(statistic=69.79031173749213, pvalue=6.595604283943106e-17, dof=1, expected_freq=array([[6788.04706471,  226.95293529],
       [6760.95293529,  226.04706471]]))
chi2-statistic: 69.79, p-value: 0.0
The difference is statistically significant, Null Hypothesis is rejected.


In [11]:
from scipy import stats

test_data = pd.DataFrame(data={'test_group': ['basic']*7015 + ['test']*6987,
                               'conversion': [3.5]*100 + [9.99] * 29 + [24.99]*10 + [0]*(7015-139) + [3.5]*120 + [9.99]*10 + [24.99]*4+ [0]*(6987-134)})

# Перевіримо, чи все коректно згенерували:
print(test_data.groupby('test_group').describe())

alpha = 0.05

observed = pd.crosstab(test_data['test_group'].values, test_data['conversion'].values)

print(observed)

print(stats.chi2_contingency(observed))

statistic, p_value, dof, expected_values = stats.chi2_contingency(observed)

print(f'chi2-statistic: {round(statistic, 2)}, p-value: {round(pvalue, 2)}')

if p_value < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')

           conversion                                               
                count      mean       std  min  25%  50%  75%    max
test_group                                                          
basic          7015.0  0.126815  1.208950  0.0  0.0  0.0  0.0  24.99
test           6987.0  0.088716  0.838438  0.0  0.0  0.0  0.0  24.99
col_0  0.00   3.50   9.99   24.99
row_0                            
basic   6876    100     29     10
test    6853    120     10      4
Chi2ContingencyResult(statistic=13.628614719261734, pvalue=0.0034568372980170237, dof=3, expected_freq=array([[6878.22703899,  110.21996858,   19.53899443,    7.013998  ],
       [6850.77296101,  109.78003142,   19.46100557,    6.986002  ]]))
chi2-statistic: 13.63, p-value: 0.0
The difference is statistically significant, Null Hypothesis is rejected.


In [12]:
#https://www.kaggle.com/datasets/yufengsui/mobile-games-ab-testing/code

data = pd.read_csv("/Users/olenaskrypka/Downloads/cookie_cats.csv")
data

basic = data[data['version']=="gate_30"]['sum_gamerounds']
test = data[data['version']=="gate_40"]['sum_gamerounds']

alpha = 0.05

observed = pd.crosstab(data['version'].values, data['sum_gamerounds'].values)

print(observed)

print(stats.chi2_contingency(observed))

statistic, pvalue, dof, expected_values = stats.chi2_contingency(observed)

print(f'chi2-statistic: {round(statistic, 2)}, p-value: {round(pvalue, 2)}')

if pvalue < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')

col_0    0      1      2      3      4      5      6      7      8      9      \
row_0                                                                           
gate_30   1937   2749   2198   1899   1831   1442   1420   1199   1162    998   
gate_40   2057   2789   2408   2059   1798   1550   1441   1180   1105   1015   

col_0    ...  2015   2063   2124   2156   2251   2294   2438   2640   2961   \
row_0    ...                                                                  
gate_30  ...      0      0      0      1      1      0      1      0      1   
gate_40  ...      1      1      1      0      0      1      0      1      0   

col_0    49854  
row_0           
gate_30      1  
gate_40      0  

[2 rows x 942 columns]
Chi2ContingencyResult(statistic=985.4980544212584, pvalue=0.1526113569373539, dof=941, expected_freq=array([[1.97952965e+03, 2.74477597e+03, 2.28285268e+03, ...,
        4.95625852e-01, 4.95625852e-01, 4.95625852e-01],
       [2.01447035e+03, 2.79322403e+03, 2.32314

### Критерій Стʼюдента

scipy.stats.ttest_ind — Тест (Тест Стʼюдента) для двох незалежних вибірок.



### Нульова гіпотеза: середні двох незалежних вибірок не відрізняються.



Основні параметри функції:

* a, b — масиви значень у двох вибірках;
* axis — напрямок обчислень у випадку використання багатовимірних масивів;
* equal_var — параметр, який вказує, чи припускається рівність дисперсій. За замовчуванням, True;
* alternative — визначає альтернативну гіпотезу.
     * less — середнє вибірки a менше за середнє вибірки b - Наша альтернатива;
     * greater — середнє вибірки a більше за середнє вибірки b - Наша альтернатива;
     * two-sided — середні вибірок відрізняються - Наша альтернатива.


In [ ]:
test_data = pd.DataFrame(data={'test_group': ['basic']*7015 + ['test']*6987 ,
                               'conversion': [1]*139 + [0]*(7015-139) + [1]*314 + [0]*(6987-314)})

print(test_data)
# Перевіримо, чи все коректно згенерували:
print(test_data.groupby('test_group').describe())

In [ ]:
from scipy import stats

alpha = 0.01

statistic, pvalue = stats.ttest_ind(
                                    test_data[test_data['test_group'] == 'test']['conversion'], #a-test
                                    test_data[test_data['test_group'] == 'basic']['conversion'], #b-basic
                                    alternative='greater',
                                    equal_var = False)

print(f't-statistic: {round(statistic, 2)}, p-value: {round(pvalue, 3)}')

if pvalue < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')

<h3>Permutation test</h3>

### Тести з перестановками
Permutation test, або тест з перестановками — метод статистичного тестування, який базується на перемішуванні даних між групами для оцінки статистичної значущості результатів.



Основна ідея полягає в тому, щоб за допомогою перестановок зрозуміти, з якою ймовірністю ми отримаємо такі самі або більш екстремальні результати, якщо нульова гіпотеза справджується. Для цього проводиться багато перемішувань, для кожного з яких обчислюється статистика, а потім ми порівнюємо фактичну статистику з отриманим розподілом.



Алгоритм, якби ми обрали критерій Стʼюдента, виглядає приблизно так:

1. Розраховуємо t-статистику на тих даних, які маємо. Це і буде наша фактична статистика t0;
2. Тепер обʼєднаємо всі спостереження в одну групу й розділимо на дві рівних заново випадковим чином;
3. Розрахуємо t-статистику на цих нових двох групах;
4. Повертаємось до кроку 2, потім — до кроку 3. Робимо все те ж саме n разів;
5. Тепер усі отримані t-статистики упорядковуємо у зростаючому порядку — це наш емпіричний розподіл.


Тепер якщо наше t0 не потрапляє в середні 95% значень емпіричного розподілу, ми можемо відкидати нульову гіпотезу з імовірністю 95%.

In [ ]:
test_data = pd.DataFrame(data={'test_group': ['basic']*7015 + ['test']*6987 ,
                               'conversion': [1]*139 + [0]*(7015-139) + [1]*314 + [0]*(6987-314)})

from scipy import stats

def statistic_ttest(x, y):
    return stats.ttest_ind(x, y).statistic

alpha = 0.05
    
x = test_data[test_data['test_group']=='basic']['conversion']
y = test_data[test_data['test_group']=='test']['conversion']

results = stats.permutation_test((x, y), statistic_ttest, n_resamples=500, alternative='less',)

print(results)

print(f'statistic: {round(results.statistic, 2)}, p-value: {round(results.pvalue, 2)}')

if results.pvalue < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize = (10,6))
sns.histplot(results.null_distribution)
plt.show()

In [ ]:
from scipy import stats

def statistic_xi2(x,y):
    observed = pd.crosstab(x,y)
    return stats.chi2_contingency(observed)[0]  # chi2 statistic

alpha = 0.05

results = stats.permutation_test((test_data['test_group'].values, test_data['conversion'].values), 
                                 statistic_xi2, 
                                 n_resamples=500)

print(f'statistic: {round(results.statistic, 2)}, p-value: {round(results.pvalue, 2)}')

print(results)

if results.pvalue < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize = (10,6))
sns.histplot(results.null_distribution)
plt.show()

<h3>Visualization</h3>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.barplot(x=test_data['test_group'], 
            y=test_data['conversion'], 
            ci = 95 # Confidence Intervals для значення алфа = 5%
           )

plt.title('A/B Test Results')
plt.xlabel('Group')
plt.ylabel('Mean')

plt.show()

In [ ]:
?sns.barplot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.barplot(x=test_data['test_group'], 
            y=test_data['conversion'], 
           errorbar ='sd') # Для стандартної помилки задаємо sd, а не число

plt.title('Mean Comparison with Standart Error')
plt.xlabel('Group')
plt.ylabel('Mean')

plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.kdeplot(stats.norm.rvs(size=1000))
sns.kdeplot(stats.norm.rvs(size=1000))

plt.title('Distribution of A/B Groups')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.legend(['A', 'B'])
plt.show()

plt.figure(figsize=(10, 6))

sns.histplot(stats.norm.rvs(size=1000))
sns.histplot(stats.norm.rvs(size=1000))

plt.title('Histogram of A/B Groups')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.legend(['A', 'B'])
plt.show()


In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.kdeplot(sample_average_A)
sns.kdeplot(sample_average_B)

plt.title('Distribution of A/B Groups')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.legend(['A', 'B'])
plt.show()

plt.figure(figsize=(10, 6))

sns.histplot(sample_average_A)
sns.histplot(sample_average_B)

plt.title('Histogram of A/B Groups')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.legend(['A', 'B'])
plt.show()

In [ ]:
import seaborn as sns

# Перемішуємо дані, оскільки зараз вони упорядковані за значенням конверсії
# Якби ми використовували реальні дані - тут мало б бути сортування за датою та часом
test_data = test_data.sample(frac=1).reset_index(drop=True)

# Рахуємо кумулятивне середнє - це і є зміна конверсії з плином часу
cumulative_metric_a = test_data[test_data['test_group'] == 'basic']['conversion'].expanding().mean().reset_index(drop=True)
cumulative_metric_b = test_data[test_data['test_group'] == 'test']['conversion'].expanding().mean().reset_index(drop=True)

plt.figure(figsize=(10, 6))
plt.plot(cumulative_metric_a, label='A')
plt.plot(cumulative_metric_b, label='B')

plt.title('Cumulative Сonversion Rate Comparison')
plt.xlabel('Time')
plt.ylabel('Cumulative Сonversion Rate')

plt.legend()
plt.show()

In [ ]:
test_data = pd.DataFrame(data={'test_group': ['basic']*7015 + ['test']*6987 ,
                               'conversion': [1]*139 + [0]*(7015-139) + [1]*314 + [0]*(6987-314)})

# Рахуємо кумулятивне середнє - це і є зміна конверсії з плином часу
cumulative_metric_a = test_data[test_data['test_group'] == 'basic']['conversion'].expanding().mean().reset_index(drop=True)
cumulative_metric_b = test_data[test_data['test_group'] == 'test']['conversion'].expanding().mean().reset_index(drop=True)

plt.figure(figsize=(10, 6))
plt.plot(cumulative_metric_a, label='A')
plt.plot(cumulative_metric_b, label='B')

plt.title('Cumulative Сonversion Rate Comparison')
plt.xlabel('Time')
plt.ylabel('Cumulative Сonversion Rate')

plt.legend()
plt.show()

In [ ]:
from scipy.stats import shapiro, anderson, kstest, normaltest

# Тест Шапіро-Уілка
shapiro_test = shapiro(test_data['conversion'].values)
print(f'Тест Шапіро-Уілка: статистика={shapiro_test.statistic}, p-значення={shapiro_test.pvalue}')

# Тест Андерсона-Дарлінга
anderson_test = anderson(test_data['conversion'].values)
print(f'Тест Андерсона-Дарлінга: статистика={anderson_test.statistic}, критичні значення={anderson_test.critical_values}')

# Тест Колмогорова-Смирнова
kstest_test = kstest(test_data['conversion'].values, 'norm')
print(f'Тест Колмогорова-Смирнова: статистика={kstest_test.statistic}, p-значення={kstest_test.pvalue}')

# Тест Д'Агостіно
dagostino_test = normaltest(test_data['conversion'].values)
print(f'Тест Д\'Агостіно: статистика={dagostino_test.statistic}, p-значення={dagostino_test.pvalue}')